# AI Talking Photo — MuseTalk 1.5 高品質 Colab

單張人物照片 + 繁體中文講稿 → Edge TTS → MuseTalk 1.5 → MP4。

**使用前請先在 Colab 選擇 GPU 執行階段，再依序執行全部儲存格。** MuseTalk 1.5 在本專案要求至少 4GB 顯示記憶體；GTX 1050 2GB 請繼續使用 Wav2Lip。

> 最後會建立臨時 `gradio.live` 網址。請只使用你有權使用的人像與內容，不要在公開連結處理敏感素材。

In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('未偵測到 NVIDIA GPU。請在「執行階段 → 變更執行階段類型」選擇 GPU，再重新全部執行。')
subprocess.run(['nvidia-smi'], check=True)

## 1. 取得最新專案
每次從 GitHub `main` 取得最新程式；若同一個 Colab 工作階段已存在專案，就更新並重設到 `origin/main`。

In [ ]:
from pathlib import Path
import os, subprocess
REPO_URL = 'https://github.com/similaitw/ai-talking-photo.git'
REPO = Path('/content/ai-talking-photo')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

## 2. 建立主程式 Python 3.11 環境
主 Gradio / Edge TTS 使用獨立 Python 3.11；MuseTalk 會另外建立自己的 Python 3.10 環境，兩者不互相污染。

In [ ]:
import shutil, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
UV = shutil.which('uv')
if not UV:
    raise RuntimeError('uv 安裝失敗。')
subprocess.run([UV, 'python', 'install', '3.11'], check=True)
MAIN_VENV = REPO / '.colab-venv'
if not MAIN_VENV.exists():
    subprocess.run([UV, 'venv', '--python', '3.11', '--seed', str(MAIN_VENV)], check=True)
MAIN_PYTHON = str(MAIN_VENV / 'bin' / 'python')
subprocess.run([MAIN_PYTHON, '-m', 'pip', 'install', 'torch==2.5.1', '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
subprocess.run([MAIN_PYTHON, '-m', 'pip', 'install', '-r', str(REPO / 'requirements.txt')], check=True)
subprocess.run([MAIN_PYTHON, '-c', "import torch; print('Main PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())"], check=True)

## 3. 安裝 MuseTalk 1.5 與模型
這一步會建立 `.venv-musetalk`、固定官方 MuseTalk 版本、安裝官方建議的 PyTorch / MMLab 套件並下載推論模型。第一次執行需要下載較多資料。

In [ ]:
subprocess.run(['bash', str(REPO / 'scripts' / 'setup_musetalk_colab.sh')], cwd=REPO, check=True)

## 4. 啟動高品質 Gradio
開啟輸出的 `gradio.live` 網址。預設嘴型引擎會選 **MuseTalk 1.5（高品質／建議 Colab）**。第一次比較建議把 GFPGAN 留在「關閉」，先單獨比較嘴型品質。

此儲存格會持續執行以維持網站；完成測試後可按停止。

In [ ]:
env = os.environ.copy()
env['TALKING_PHOTO_DEFAULT_BACKEND'] = 'musetalk'
subprocess.run([
    MAIN_PYTHON, '-c',
    'from app import build_app; build_app().launch(share=True)'
], cwd=REPO, env=env, check=True)